# Leakage-Safe Grouped Train/Validation/Test Split

This notebook creates leakage-safe grouped train/validation/test splits for the thesis modeling dataset. Because the same spare-part listing can appear repeatedly across scrape dates, all observations belonging to the same listing must remain in exactly one split.

The notebook performs only split preparation and validation. It does **not** do feature engineering, encoding, scaling, imputation, or modeling preprocessing.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

## Load Dataset

The modeling dataset is loaded defensively so that column names are standardized immediately after reading. This makes downstream split logic more robust and easier to maintain.

In [2]:
data_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/clean_master_dataset.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path, skipinitialspace=True)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

if "scrape_date" in df.columns:
    df["scrape_date"] = pd.to_datetime(df["scrape_date"], errors="coerce")

print(f"Dataset path: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print()
print("Column preview:")
print(df.columns.tolist())
print()

display(df.head())

Dataset path: /Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/clean_master_dataset.csv
Rows: 11,374
Columns: 56

Column preview:
['product_id', 'part_name', 'price', 'quality_grade', 'oem_number', 'mileage', 'brand', 'model', 'category', 'subcategory', 'scrape_date', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_merge_key', 'model_merge_key', 'repair_status', 'brand_is_known_model_family', 'model_looks_like_part_taxonomy', 'model_family_clean', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_total_registered', 'brand_median_vehicle_age', 'b

,product_id,part_name,price,quality_grade,oem_number,mileage,brand,model,category,subcategory,scrape_date,year_start,year_end,year_span,year_mid,brand_merge_key,model_merge_key,repair_status,brand_is_known_model_family,model_looks_like_part_taxonomy,model_family_clean,model_total_registered,model_median_vehicle_age,model_mean_vehicle_age,model_median_mileage,model_mean_mileage,model_median_engine_cc,model_median_power_kw,model_median_mass_kg,model_share_of_market,model_share_within_brand,model_share_over_10y,model_share_over_200k_km,model_automatic_share,model_petrol_share,model_diesel_share,model_ev_share,model_hybrid_share,model_turbo_share,brand_total_registered,brand_median_vehicle_age,brand_mean_vehicle_age,brand_median_mileage,brand_mean_mileage,brand_median_engine_cc,brand_median_power_kw,brand_median_mass_kg,brand_share_of_market,brand_share_over_10y,brand_share_over_200k_km,brand_automatic_share,brand_petrol_share,brand_diesel_share,brand_ev_share,brand_hybrid_share,brand_turbo_share
0,63136980,"Contact roll Airbag - , e-",177.600,A2,FI02042722A,"224,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
1,64390201,"Contact roll Airbag - , e-",177.600,A2,FI05351686A,"272,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
2,58952159,"Contact roll Airbag - , e-",177.600,A2,FI27837687A,"134,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
3,63225719,"Contact roll Airbag - , e-",177.600,A2,FI27837687A,"253,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686
4,64336640,"Contact roll Airbag - , e-",177.600,A2,FI09389104A,"152,000.000",vw,golf,airbag,contact roll airbag,2026-02-03,2013,2020,7,"2,016.500",volkswagen,golf,original_valid,True,False,golf,86762,13.000,13.749,"186,801.000","195,467.968","1,395.000",81.000,"1,330.000",0.034,0.311,0.652,0.433,0.383,0.786,0.170,0.013,0.068,0.726,279311,11.000,12.154,"177,745.000","192,564.366","1,498.000",85.000,"1,400.000",0.109,0.538,0.387,0.478,0.657,0.246,0.075,0.073,0.686


## Key Variable Definitions

The target, group identifier, and optional date column are defined explicitly here so the notebook is easy to edit later if the dataset structure changes.

In [3]:
target_column = "price"
group_column = "product_id"
date_column = "scrape_date" if "scrape_date" in df.columns else None

candidate_group_columns = [
    column for column in df.columns
    if column.endswith("_id") or column in {"product_id", "listing_id", "item_id", "group_id"}
]

id_or_leakage_columns = [
    group_column,
    "oem_number",
]
id_or_leakage_columns = [column for column in id_or_leakage_columns if column in df.columns]

feature_columns = [
    column for column in df.columns
    if column not in {target_column, *id_or_leakage_columns}
]

print(f"Target column: {target_column}")
print(f"Group column: {group_column}")
print(f"Date column: {date_column}")
print(f"Candidate ID/group columns: {candidate_group_columns}")
print(f"Number of feature columns: {len(feature_columns)}")

Target column: price
Group column: product_id
Date column: scrape_date
Candidate ID/group columns: ['product_id']
Number of feature columns: 53


## Validation Checks Before Splitting

These checks confirm that the required columns exist and that the grouped split is meaningful. If a required column is missing, the notebook stops immediately with a clear error.

In [4]:
required_columns = [target_column, group_column]
missing_required_columns = [column for column in required_columns if column not in df.columns]
if missing_required_columns:
    raise KeyError(f"Missing required columns: {missing_required_columns}")

if df[target_column].isna().any():
    raise ValueError(f"Target column '{target_column}' contains missing values.")
if df[group_column].isna().any():
    raise ValueError(f"Group column '{group_column}' contains missing values.")

group_sizes = df.groupby(group_column).size().rename("group_size")

validation_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_groups",
            "average_observations_per_group",
            "median_group_size",
            "max_group_size",
        ],
        "value": [
            len(df),
            len(df.columns),
            df[group_column].nunique(),
            group_sizes.mean(),
            group_sizes.median(),
            group_sizes.max(),
        ],
    }
)

display(validation_summary)
print("Top group sizes:")
display(group_sizes.sort_values(ascending=False).head(10).to_frame())

,metric,value
0,rows,"11,374.000"
1,columns,56.000
2,unique_groups,"2,619.000"
3,average_observations_per_group,4.343
4,median_group_size,6.000
5,max_group_size,7.000


Top group sizes:


,group_size
product_id,
62122634,7
53468309,7
64248645,7
57894045,7
54479020,7
57894044,7
64935321,7
53822125,7
57182287,7


## Create Leakage-Safe Grouped Split

The split is created by shuffling the unique group IDs with a fixed random seed and then assigning whole groups to train, validation, and test. This keeps every repeated observation of the same listing in exactly one split.


In [ ]:
random_state = 42
train_size = 0.70
validation_size = 0.15
test_size = 0.15

rng = np.random.default_rng(random_state)
unique_groups = df[group_column].drop_duplicates().to_numpy()
shuffled_groups = rng.permutation(unique_groups)

n_groups = len(shuffled_groups)
train_group_count = int(round(n_groups * train_size))
validation_group_count = int(round(n_groups * validation_size))
test_group_count = n_groups - train_group_count - validation_group_count

train_group_ids = set(shuffled_groups[:train_group_count])
val_group_ids = set(shuffled_groups[train_group_count:train_group_count + validation_group_count])
test_group_ids = set(shuffled_groups[train_group_count + validation_group_count:])

train_df = df[df[group_column].isin(train_group_ids)].copy()
val_df = df[df[group_column].isin(val_group_ids)].copy()
test_df = df[df[group_column].isin(test_group_ids)].copy()

print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
print(f"Test rows: {len(test_df):,}")
print()
print(f"Train groups: {len(train_group_ids):,}")
print(f"Validation groups: {len(val_group_ids):,}")
print(f"Test groups: {len(test_group_ids):,}")


Train rows: 7,997
Validation rows: 1,695
Test rows: 1,682

Train groups: 1,833
Validation groups: 393
Test groups: 393


## Strict Leakage Checks

The following checks confirm that no group ID appears in more than one split. Any overlap raises an error immediately.

In [ ]:
train_groups = set(train_df[group_column].unique())
val_groups = set(val_df[group_column].unique())
test_groups = set(test_df[group_column].unique())

train_val_overlap = train_groups & val_groups
train_test_overlap = train_groups & test_groups
val_test_overlap = val_groups & test_groups

print(f"Train/Validation overlap: {len(train_val_overlap)}")
print(f"Train/Test overlap: {len(train_test_overlap)}")
print(f"Validation/Test overlap: {len(val_test_overlap)}")

Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


## Date Diagnostics

If a date column is available, the split is checked for date coverage. This is diagnostic only; the split remains group-based rather than time-based.

In [10]:
if date_column is not None:
    date_diagnostics = pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "min_date": [train_df[date_column].min(), val_df[date_column].min(), test_df[date_column].min()],
            "max_date": [train_df[date_column].max(), val_df[date_column].max(), test_df[date_column].max()],
            "unique_dates": [train_df[date_column].nunique(), val_df[date_column].nunique(), test_df[date_column].nunique()],
        }
    )
    display(date_diagnostics)
else:
    print("No date column available for date diagnostics.")

,split,min_date,max_date,unique_dates
0,train,2026-02-03,2026-02-18,6
1,validation,2026-02-03,2026-02-18,6
2,test,2026-02-03,2026-02-18,6


## Save Split Outputs

The split datasets are saved separately so that all later preprocessing and feature engineering steps can be performed after the leakage-safe split has already been established.

In [12]:
output_dir = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/splits")
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "train_grouped.csv"
val_path = output_dir / "validation_grouped.csv"
test_path = output_dir / "test_grouped.csv"
assignment_path = output_dir / "group_split_assignment.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

assignment_df = pd.concat(
    [
        train_df[[group_column]].drop_duplicates().assign(split="train"),
        val_df[[group_column]].drop_duplicates().assign(split="validation"),
        test_df[[group_column]].drop_duplicates().assign(split="test"),
    ],
    ignore_index=True,
)
assignment_df.to_csv(assignment_path, index=False)

## Summary

Grouped splitting has been completed. The same listing group is kept entirely within one split.